# Data quality scan: buildings with `HtTot` (Oslo + Drammen)

Goal: decide, from facts, which buildings go into the silver/gold layers for predicting `HtTot` from weather + time.
Run the cells top to bottom. Each step ends with a **Decision** note so you can write down your thresholds before preprocessing.

Only the header rows are read in steps 1-2. From step 3 the data is loaded building by building.

In [12]:
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

ROOT = Path("..") / "datasets"
SOURCES = {
    "oslo": ROOT / "COFACTOR-SBHUB-Oslo" / "Dataset_buildings_csv_files",
    "drammen": ROOT / "Cofactor_Drammen_Buildings_45_v3",
}
WEATHER = ["Tout", "SolGlob", "WindSpd", "WindDir", "RH"]

## 1. Read only the metadata + column row of every file
Line 1 is `Header_line;N`, and `N` is the line number of the column row. Lines 2 up to the blank line are `key;value` metadata.

In [13]:
def read_head(path):
    """Return (meta dict, column list, header line number) without reading the data rows."""
    meta, cols = {}, None
    with open(path, encoding="utf-8", errors="replace") as f:
        n = int(f.readline().strip().split(";")[1])
        for i in range(2, n + 1):
            line = f.readline().rstrip("\r\n")
            if i == n:
                cols = line.split(";")
            elif ";" in line and line.split(";")[0]:
                k, v = line.split(";", 1)
                meta[k] = v
    return meta, cols, n

rows = []
for city, folder in SOURCES.items():
    for p in sorted(folder.glob("*.txt")):
        meta, cols, n = read_head(p)
        rows.append({"city": city, "file": p.name, "path": p, "header_line": n,
                     "cols": cols, **meta})
files = pd.DataFrame(rows)
try:
    files["building_id"] = files["building_id"].astype(int)
except ValueError as e:
    print("building_id is not a clean integer:", e)
    bad = files[~files["building_id"].astype(str).str.fullmatch(r"\d+")]
    print(f"\n{len(bad)} file(s) with a bad building_id:")
    print(bad[["city", "file", "building_id", "header_line"]].to_string())
    print("\nFirst rows of the dataframe:")
    print(files.head().to_string())
print(files.groupby("city").size())
files[["city", "file", "building_id", "header_line", "building_category", "year_of_construction", "floor_area"]].head()

city
drammen    45
oslo       48
dtype: int64


,city,file,building_id,header_line,building_category,year_of_construction,floor_area
0,oslo,export_8091.txt,8091,19,Sch,2004,6550
1,oslo,export_8092.txt,8092,18,Sch,1970,7047
2,oslo,export_8093.txt,8093,17,Sch,1965,9072
3,oslo,export_8094.txt,8094,18,Sch,NaN,27479
4,oslo,export_8095.txt,8095,17,Sch,1975,6066


## 2. Which buildings have `HtTot`?
Check: do all rows have a unique `building_id`, and how many buildings per city have `HtTot` in the header?

In [14]:
files["has_HtTot"] = files["cols"].apply(lambda c: "HtTot" in c)
files["has_HtDH"] = files["cols"].apply(lambda c: "HtDH" in c)
print("duplicate building_id:", files["building_id"].duplicated().sum())
print(files.groupby("city")[["has_HtTot", "has_HtDH"]].sum())
print("total with HtTot:", files["has_HtTot"].sum())

target = files[files["has_HtTot"]].reset_index(drop=True)
print("\nbuilding_category among target buildings:")
print(target.groupby(["city", "building_category"]).size())

duplicate building_id: 0
         has_HtTot  has_HtDH
city                        
drammen          5         3
oslo            33        29
total with HtTot: 38

building_category among target buildings:
city     building_category
drammen  Kdg                   2
         Nsh                   3
oslo     Sch                  33
dtype: int64


**Decision:** you counted about 37, my header scan gives 38 (33 Oslo + 5 Drammen). Step 4 will show whether one of them has an empty `HtTot`.

## 3. Common columns across the target buildings
Weather columns are expected everywhere. Check which columns are present in every target building and which are only in some.

In [15]:
from collections import Counter
cnt = Counter(c for cols in target["cols"] for c in cols)
common = pd.Series(cnt).sort_values(ascending=False).rename("n_buildings").to_frame()
common["share"] = (common["n_buildings"] / len(target)).round(2)
print(common)
print("\nin every target building:", common.index[common["n_buildings"] == len(target)].tolist())

           n_buildings  share
TimeStamp           38   1.00
HtTot               38   1.00
ElImp               38   1.00
Tout                38   1.00
WindSpd             38   1.00
SolGlob             38   1.00
WindDir             38   1.00
RH                  33   0.87
HtDH                29   0.76
HtDHW                8   0.21
HtHP                 7   0.18
ElHP                 7   0.18
ElBoil               6   0.16
ElHWH                5   0.13
ElPV                 3   0.08
ElExp                3   0.08
ElFan                2   0.05
HtBio                1   0.03
ElSpace              1   0.03
HtVent               1   0.03
ElLight              1   0.03
HtSpace              1   0.03
HtSnow               1   0.03
ClComf               1   0.03
ElEV                 1   0.03
ElDHW                1   0.03
ElSnow               1   0.03
HtOil                1   0.03

in every target building: ['TimeStamp', 'HtTot', 'ElImp', 'Tout', 'WindSpd', 'SolGlob', 'WindDir']


**Decision:** `RH` is expected to be missing in Drammen. Choose: drop `RH`, or keep it and fill/skip for Drammen.

## 4. Load each target building and build a per-building quality table
Timestamps are fixed UTC+1 all year (no DST), so the offset is stripped and times are kept naive, like in the earlier check notebook.
This cell reads the data of the target buildings only, one at a time, and keeps just a summary.

In [16]:
def load(row):
    df = pd.read_csv(row["path"], sep=";", skiprows=row["header_line"] - 1)
    raw_ts = df["TimeStamp"]
    offsets = raw_ts.str[-5:].unique().tolist()
    df["ts"] = pd.to_datetime(raw_ts.str[:19])
    df = df.set_index("ts").drop(columns="TimeStamp").sort_index()
    return df, offsets

def longest_run(mask):
    """Length of the longest run of True in a boolean Series."""
    grp = (~mask).cumsum()
    return int(mask.groupby(grp).sum().max()) if mask.any() else 0

summ = []
for _, r in target.iterrows():
    df, offsets = load(r)
    idx = df.index
    full = pd.date_range(idx.min(), idx.max(), freq="h")
    h = df["HtTot"]
    flat = h.eq(h.shift()) & h.gt(0)
    summ.append({
        "city": r["city"], "id": r["building_id"], "cat": r["building_category"],
        "rows": len(df), "start": idx.min().date(), "end": idx.max().date(),
        "years": round((idx.max() - idx.min()).days / 365.25, 2),
        "missing_hours": len(full.difference(idx)), "dup_ts": int(idx.duplicated().sum()),
        "offsets": ",".join(offsets),
        "Ht_nan_%": round(h.isna().mean() * 100, 1),
        "Ht_zero_%": round((h == 0).mean() * 100, 1),
        "Ht_neg": int((h < 0).sum()),
        "Ht_flat_%": round(flat.mean() * 100, 1),
        "Ht_longest_nan_h": longest_run(h.isna()),
        "Ht_max": h.max(),
        "Tout_nan_%": round(df["Tout"].isna().mean() * 100, 1),
        "Tout_sentinel": int((df["Tout"] < -100).sum()),
        "Sol_neg": int((df["SolGlob"] < 0).sum()),
        "RH_present": "RH" in df.columns,
        "HtTot==HtDH": bool("HtDH" in df.columns and (df["HtTot"].dropna() == df["HtDH"].reindex(df["HtTot"].dropna().index)).mean() > 0.99),
    })
quality = pd.DataFrame(summ).sort_values(["city", "id"]).reset_index(drop=True)
quality

,city,id,cat,rows,start,end,years,missing_hours,dup_ts,offsets,Ht_nan_%,Ht_zero_%,Ht_neg,Ht_flat_%,Ht_longest_nan_h,Ht_max,Tout_nan_%,Tout_sentinel,Sol_neg,RH_present,HtTot==HtDH
0,drammen,6409,Kdg,35064,2018-01-01,2021-12-31,4.00,0,0,+0100,1.1,0.1,0,18.8,264,9.040000e+04,0.2,0,406,False,False
1,drammen,6412,Nsh,35064,2018-01-01,2021-12-31,4.00,0,0,+0100,0.3,8.1,0,36.1,48,1.850530e+09,0.2,0,386,False,False
2,drammen,6429,Kdg,36708,2018-01-01,2022-03-10,4.19,0,0,+0100,50.0,0.0,0,8.1,3312,9.900000e+04,0.3,0,430,False,True
3,drammen,6436,Nsh,36341,2018-01-01,2022-02-23,4.15,0,0,+0100,4.9,29.5,0,12.7,48,3.900000e+05,0.3,0,407,False,True
4,drammen,6442,Nsh,36348,2018-01-01,2022-02-23,4.15,0,0,+0100,0.7,25.1,0,30.5,168,1.700000e+05,0.3,0,407,False,False
5,oslo,8091,Sch,24600,2021-01-01,2023-10-22,2.80,0,0,+0100,0.0,2.8,0,9.5,0,4.280000e+05,0.0,0,945,True,True
6,oslo,8092,Sch,85978,2014-01-01,2023-10-23,9.81,0,0,+0100,0.0,18.4,0,30.2,1,4.700000e+05,0.0,1,3770,True,False
7,oslo,8093,Sch,8809,2019-03-05,2020-03-06,1.00,0,0,+0100,0.0,1.3,0,16.9,0,4.256000e+05,0.0,1,386,True,False
8,oslo,8094,Sch,35088,2016-01-01,2020-01-01,4.00,0,0,+0100,0.0,28.5,0,29.0,0,1.700000e+06,0.0,0,1595,True,True
9,oslo,8095,Sch,50914,2018-01-01,2023-10-23,5.81,0,0,+0100,0.0,26.6,0,13.1,0,4.356042e+05,0.0,1,2075,True,False


Read the table column by column:
- `rows`, `years`: how much data per building.
- `missing_hours`, `dup_ts`: gaps and duplicate timestamps in the time index.
- `offsets`: should be only `+0100`. Anything else means the fixed-UTC+1 assumption is wrong for that file.
- `Ht_nan_%`, `Ht_longest_nan_h`: how empty the target is, and the longest hole.
- `Ht_zero_%`, `Ht_neg`, `Ht_flat_%`: suspicious target values (zeros, negatives, the same value repeated).
- `Tout_sentinel`, `Sol_neg`: known weather problems (-273.15 sentinel, negative solar).
- `HtTot==HtDH`: whether the target is just a copy of district heating (so `HtDH` must never be a feature).

## 5. Summary statistics over all target buildings

In [17]:
print(quality.describe().T[["min", "25%", "50%", "75%", "max"]].round(2))
print("\nbuildings with non-+0100 offsets:", (quality["offsets"] != "+0100").sum())
print("buildings with any duplicate timestamps:", (quality["dup_ts"] > 0).sum())
print("buildings where HtTot is a copy of HtDH:", quality["HtTot==HtDH"].sum())
print("total rows:", quality["rows"].sum())

                      min        25%        50%        75%           max
id                 6409.0    8095.25    8109.50    8123.50  8.137000e+03
rows               8809.0   35214.00   49080.00   94197.00  9.694600e+04
years                 1.0       4.01       5.60      10.75  1.106000e+01
missing_hours         0.0       0.00       0.00       0.00  0.000000e+00
dup_ts                0.0       0.00       0.00       0.00  0.000000e+00
Ht_nan_%              0.0       0.00       0.00       0.00  7.310000e+01
Ht_zero_%             0.0       3.98      11.25      20.08  4.510000e+01
Ht_neg                0.0       0.00       0.00       0.00  0.000000e+00
Ht_flat_%             2.4      18.82      23.40      29.45  3.610000e+01
Ht_longest_nan_h      0.0       0.00       0.50       7.00  7.087100e+04
Ht_max            90400.0  399750.00  525000.00  695750.00  1.850530e+09
Tout_nan_%            0.0       0.00       0.00       0.00  3.000000e-01
Tout_sentinel         0.0       0.00       1.00    

## 6. Flag problem buildings
Change the thresholds to your own choices, then look at which buildings fall out.

In [18]:
# EDIT THESE after reading steps 4-5
MAX_HT_NAN_PCT = 20       # target mostly empty
MAX_LONGEST_NAN_H = 24 * 30  # one hole longer than this many hours
MAX_HT_ZERO_PCT = 20      # target mostly zero
MAX_HT_FLAT_PCT = 20      # target mostly repeated values
MIN_YEARS = 1.0           # at least a full seasonal cycle

q = quality.copy()
q["flag_nan"] = q["Ht_nan_%"] > MAX_HT_NAN_PCT
q["flag_hole"] = q["Ht_longest_nan_h"] > MAX_LONGEST_NAN_H
q["flag_zero"] = q["Ht_zero_%"] > MAX_HT_ZERO_PCT
q["flag_flat"] = q["Ht_flat_%"] > MAX_HT_FLAT_PCT
q["flag_short"] = q["years"] < MIN_YEARS
flag_cols = [c for c in q.columns if c.startswith("flag_")]
q["n_flags"] = q[flag_cols].sum(axis=1)
print("flagged:", (q["n_flags"] > 0).sum(), "of", len(q))
q[q["n_flags"] > 0][["city", "id", "years", "Ht_nan_%", "Ht_longest_nan_h", "Ht_zero_%", "Ht_flat_%"] + flag_cols]

flagged: 32 of 38


,city,id,years,Ht_nan_%,Ht_longest_nan_h,Ht_zero_%,Ht_flat_%,flag_nan,flag_hole,flag_zero,flag_flat,flag_short
1,drammen,6412,4.00,0.3,48,8.1,36.1,False,False,False,True,False
2,drammen,6429,4.19,50.0,3312,0.0,8.1,True,True,False,False,False
3,drammen,6436,4.15,4.9,48,29.5,12.7,False,False,True,False,False
4,drammen,6442,4.15,0.7,168,25.1,30.5,False,False,True,True,False
6,oslo,8092,9.81,0.0,1,18.4,30.2,False,False,False,True,False
8,oslo,8094,4.00,0.0,0,28.5,29.0,False,False,True,True,False
9,oslo,8095,5.81,0.0,0,26.6,13.1,False,False,True,False,False
10,oslo,8096,11.06,0.0,0,14.0,33.5,False,False,False,True,False
11,oslo,8102,3.81,0.0,1,7.6,21.2,False,False,False,True,False
13,oslo,8104,4.81,0.0,10,8.7,33.6,False,False,False,True,False


## 7. Look at the target by eye
Monthly total `HtTot` (MWh) per building. A building should show high winter and low summer values. Odd shapes (flat, all zero, big jumps) deserve a closer look.

In [19]:
monthly = {}
for _, r in target.iterrows():
    df, _ = load(r)
    monthly[f'{r["city"]}-{r["building_id"]}'] = (df["HtTot"].resample("MS").sum(min_count=1) / 1e6).round(1)
pd.DataFrame(monthly).T

ts,2012-10-01,2012-11-01,2012-12-01,2013-01-01,2013-02-01,2013-03-01,2013-04-01,2013-05-01,2013-06-01,2013-07-01,2013-08-01,2013-09-01,2013-10-01,2013-11-01,2013-12-01,2014-01-01,2014-02-01,2014-03-01,2014-04-01,2014-05-01,2014-06-01,2014-07-01,2014-08-01,2014-09-01,2014-10-01,...,2021-10-01,2021-11-01,2021-12-01,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,2022-11-01,2022-12-01,2023-01-01,2023-02-01,2023-03-01,2023-04-01,2023-05-01,2023-06-01,2023-07-01,2023-08-01,2023-09-01,2023-10-01
oslo-8091,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.7,79.6,111.7,87.3,77.3,72.0,56.0,34.0,9.4,5.5,8.8,26.1,40.3,58.9,105.6,85.6,69.7,99.4,60.9,32.6,8.0,10.5,12.3,22.9,41.5
oslo-8092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99.1,78.9,66.2,48.1,27.6,8.1,1.9,9.0,20.8,48.6,...,23.5,48.6,66.1,61.8,48.9,46.7,26.0,13.7,6.2,4.0,3.2,14.3,27.6,41.9,72.7,68.0,50.1,58.6,32.8,18.9,3.6,1.1,4.1,12.7,22.9
oslo-8093,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oslo-8094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oslo-8095,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,30.1,49.0,68.0,62.2,54.2,53.2,34.4,19.3,8.0,4.6,6.1,21.4,32.2,46.0,62.9,55.5,42.6,51.5,32.1,19.4,3.9,7.8,8.6,10.6,20.7
oslo-8096,90.0,100.4,158.6,155.8,117.2,113.3,62.4,22.8,10.7,1.1,1.4,29.1,54.0,98.9,120.4,132.1,107.5,117.1,74.0,42.4,8.2,7.4,10.5,22.2,49.0,...,41.4,82.4,120.5,117.0,90.7,76.4,46.8,21.0,5.9,0.0,0.2,11.5,36.3,67.3,117.9,108.0,84.5,95.4,53.2,27.5,4.8,4.3,8.0,7.4,24.6
oslo-8102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,80.5,157.2,176.6,145.6,106.4,125.9,77.3,48.8,18.7,13.2,16.1,54.1,79.1,142.8,243.6,226.3,183.8,224.7,78.2,32.8,11.3,11.9,14.9,23.5,41.2
oslo-8103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,92.7,150.7,174.6,155.1,118.5,106.6,65.2,34.7,10.7,5.5,10.9,33.5,62.2,94.0,157.4,142.2,108.0,124.6,61.1,27.8,9.2,11.3,14.8,20.5,42.4
oslo-8104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,32.2,49.8,71.0,62.6,52.6,48.8,32.4,17.0,5.2,3.6,11.3,22.3,32.0,44.7,69.6,64.6,51.4,60.0,31.1,16.5,4.5,4.2,11.8,17.0,23.2
oslo-8105,91.7,113.0,209.8,206.8,159.2,152.1,81.7,27.6,11.0,4.0,5.9,27.5,62.6,105.8,122.7,185.4,115.2,91.4,41.7,20.6,5.3,2.2,5.6,13.9,44.6,...,48.4,85.1,136.2,112.7,97.7,98.8,44.8,18.8,5.8,3.2,3.5,17.8,43.3,72.8,133.7,119.7,88.2,102.7,45.5,17.5,4.2,3.6,4.6,10.4,31.8


## 8. Relationship to weather
Correlation of `HtTot` with outdoor temperature per building. A strongly negative value is expected (colder, more heat). A value near zero suggests the building is hard to predict from weather alone or the data is broken.

In [20]:
corr = {}
for _, r in target.iterrows():
    df, _ = load(r)
    d = df[df["Tout"] > -100]
    corr[f'{r["city"]}-{r["building_id"]}'] = d["HtTot"].corr(d["Tout"])
pd.Series(corr, name="corr(HtTot, Tout)").round(2).sort_values()

oslo-8132      -0.88
oslo-8116      -0.87
oslo-8131      -0.86
oslo-8105      -0.85
oslo-8128      -0.84
drammen-6429   -0.84
oslo-8103      -0.83
oslo-8127      -0.82
drammen-6436   -0.81
drammen-6442   -0.81
oslo-8126      -0.80
oslo-8113      -0.80
oslo-8121      -0.79
oslo-8122      -0.79
oslo-8133      -0.79
oslo-8108      -0.77
oslo-8091      -0.77
oslo-8137      -0.76
oslo-8096      -0.76
oslo-8119      -0.75
oslo-8094      -0.73
oslo-8106      -0.73
oslo-8114      -0.73
oslo-8117      -0.71
oslo-8124      -0.71
oslo-8120      -0.69
oslo-8107      -0.69
oslo-8092      -0.68
oslo-8125      -0.67
oslo-8095      -0.66
oslo-8102      -0.66
oslo-8104      -0.65
oslo-8110      -0.64
oslo-8093      -0.64
oslo-8135      -0.57
oslo-8109      -0.50
drammen-6409   -0.29
drammen-6412   -0.01
Name: corr(HtTot, Tout), dtype: float64

## 9. Write down your decisions
Fill these in before the preprocessing step.

- Buildings to exclude (id and reason):
- `RH`: drop / keep:
- Gap rule for weather (interpolate up to N hours?):
- Gap rule for `HtTot` (never fill, drop those rows):
- Handling of `HtTot == 0` and flat stretches:
- Train / validation / test split by building (which ids go where):